In [1]:
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/data/processed"

Mounted at /content/drive
vihallu-train-split.jsonl  vihallu-val-split.jsonl


In [2]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers

import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding, get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

from huggingface_hub import login
from dotenv import load_dotenv
import os
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

os.makedirs("/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/model", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Using device: cpu


In [3]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

train_df = load_jsonl("/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/data/processed/vihallu-train-split.jsonl")
val_df = load_jsonl("/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/data/processed/vihallu-val-split.jsonl")
test_df = load_jsonl("/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/data/jsonl/vihallu-public-test.jsonl")

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))

# Encode label
labels = sorted(train_df["label"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

train_df["label"] = train_df["label"].map(label2id)
val_df["label"]   = val_df["label"].map(label2id)

Train/Val/Test: 6300 700 1000


In [4]:
tokenizer = AutoTokenizer.from_pretrained("uitnlp/CafeBERT")
specials = ["<PROMPT>", "</PROMPT>", "<CONTEXT>", "</CONTEXT>", "<RESPONSE>", "</RESPONSE>"]
tokenizer.add_special_tokens({"additional_special_tokens": specials})

from torch.utils.data import Dataset
import torch
import numpy as np

class ResponseDataset(Dataset):
    def __init__(self, df, tokenizer, label2id=None, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label2id = label2id or {"extrinsic": 0, "intrinsic": 1, "no": 2}
        self.max_length = max_length

    def _make_premise(self, context):
        return f"<CONTEXT> {context}"

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        context, prompt, response = str(row.get("context", "")), str(row.get("prompt", "")), str(row.get("response", ""))

        raw_label = row.get("label", None)
        if isinstance(raw_label, (int, np.integer)):
            label = int(raw_label)
        else:
            label = self.label2id.get(str(raw_label).lower(), -1)

        premise = self._make_premise(context)
        text = f"{premise} [SEP] {prompt} [SEP] {response}"
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

    def __len__(self):
        return len(self.df)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(ResponseDataset(train_df, tokenizer, label2id=label2id), batch_size=8, shuffle=True, collate_fn=collator)
val_loader = DataLoader(ResponseDataset(val_df, tokenizer, label2id=label2id), batch_size=8, collate_fn=collator)
test_loader = DataLoader(ResponseDataset(test_df, tokenizer, label2id=None), batch_size=8, collate_fn=collator)

tokenizer_config.json:   0%|          | 0.00/496 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [5]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

class CafeBERTNLIClassifier(nn.Module):
    def __init__(self, num_labels=len(label2id)):
        super().__init__()
        self.base = AutoModel.from_pretrained("uitnlp/CafeBERT")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.base.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        x = mean_pool(outputs.last_hidden_state, attention_mask)
        x = self.dropout(x)
        return self.fc(x)

model = CafeBERTNLIClassifier(num_labels=len(label2id)).to(device)
model.base.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaModel were not initialized from the model checkpoint at uitnlp/CafeBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(250008, 1024, padding_idx=1)

In [ ]:
def evaluate(model, loader, criterion, device, id2label=None):
    model.eval()
    total_loss, preds, trues = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch.get("token_type_ids", None)
            if token_type_ids is not None: token_type_ids = token_type_ids.to(device)
            labels = batch.get("labels", None)
            if labels is not None:
                labels = labels.to(device)
                logits = model(input_ids, attention_mask, token_type_ids)
                loss = criterion(logits, labels)
                total_loss += loss.item()
                preds.extend(logits.argmax(dim=1).cpu().numpy())
                trues.extend(labels.cpu().numpy())
    avg_loss = total_loss / max(1, len(loader))
    f1 = f1_score(trues, preds, average="macro") if trues else 0.0
    return avg_loss, f1

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
EPOCHS = 10
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*EPOCHS)
scaler = GradScaler("cuda", enabled=torch.cuda.is_available())

best_f1, patience, patience_counter = 0.0, 3, 0
best_path = "/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/model/best_cafebase_nli.pt"

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch.get("token_type_ids", None)
        if token_type_ids is not None: token_type_ids = token_type_ids.to(device)
        labels = batch["labels"].to(device)

        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids, attention_mask, token_type_ids)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    val_loss, val_f1 = evaluate(model, val_loader, criterion, device, id2label=id2label)
    print(f"Epoch {epoch} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save({"state_dict": model.state_dict(), "label2id": label2id}, best_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping!")
            break

Epoch 1 | Val Loss: 0.8476 | Val F1: 0.6822
Epoch 2 | Val Loss: 0.6910 | Val F1: 0.7595
Epoch 3 | Val Loss: 0.6585 | Val F1: 0.7862
Epoch 4 | Val Loss: 0.6192 | Val F1: 0.7818
Epoch 5 | Val Loss: 0.7492 | Val F1: 0.7752
Epoch 6 | Val Loss: 0.9499 | Val F1: 0.7867
Epoch 7 | Val Loss: 1.6433 | Val F1: 0.7819
Epoch 8 | Val Loss: 2.6245 | Val F1: 0.7705
Epoch 9 | Val Loss: 2.5072 | Val F1: 0.7812
Early stopping!


In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

pred_labels = []
for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    token_type_ids = batch.get("token_type_ids", None)
    if token_type_ids is not None: token_type_ids = token_type_ids.to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask, token_type_ids)
        pred_ids = torch.argmax(logits, dim=1).cpu().numpy()
        pred_labels.extend([id2label[i] for i in pred_ids])

df_output = pd.DataFrame({"id": test_df["id"], "predict_label": pred_labels})
df_output.to_csv("/content/drive/MyDrive/Colab Notebooks/vietnamese-llm-hallucination-detection/result/predicted.csv", index=False)
print("Saved predictions to predicted.csv")

Saved predictions to predicted.csv
